# Phase 2 — Shopee Preprocessing, Unrelated/Noise Filtering, and Translation for 14-Aspect DeBERTa Auto-Labeling

In [1]:
import re
import json
import math
import html
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
try:
    from ftfy import fix_text as ftfy_fix_text
    FTFY_AVAILABLE = True
except Exception as e:
    FTFY_AVAILABLE = False
    print("WARNING: ftfy is not installed. Text fixing will use a simple fallback.")
    print("Fix with: python -m pip install -U ftfy")
    print("Original error:", repr(e))

    def ftfy_fix_text(x):
        return str(x)

pd.set_option("display.max_colwidth", 180)
tqdm.pandas()

# Language detector: correct PyPI package is lingua-language-detector.
try:
    from lingua import Language, LanguageDetectorBuilder
    LINGUA_AVAILABLE = True
except Exception as e:
    LINGUA_AVAILABLE = False
    LINGUA_IMPORT_ERROR = repr(e)
    print("WARNING: Lingua language detector is not available or wrong package is installed.")
    print("Fix with:")
    print("python -m pip uninstall -y lingua lingua-language-detector")
    print("python -m pip install -U lingua-language-detector")
    print("Original error:", LINGUA_IMPORT_ERROR)

print("Pandas:", pd.__version__)

Pandas: 3.0.3


## 1. Paths and run switches

In [2]:
# =========================
# Input / output paths
# =========================
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV_PATH = Path(r"C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\shopee_google_play_reviews_raw.csv")

# Fallback for this generated notebook if you run it beside an uploaded file.
if not RAW_CSV_PATH.exists():
    fallback_path = Path("shopee_google_play_reviews_raw.csv")
    if fallback_path.exists():
        RAW_CSV_PATH = fallback_path

# =========================
# Run switches
# =========================
RUN_TRANSLATION = True
SMOKE_TEST_ROWS = None  # e.g. 100 for quick testing, None for full dataset

# Filtering behavior.
# This notebook intentionally DOES NOT filter by focused aspect keywords.
# It only removes duplicate/noisy/unrelated rows before translation and DeBERTa auto-labeling.
MIN_REVIEW_WORDS = 3
MIN_REVIEW_CHARS = 12

# Translation model.
# Helsinki-NLP/opus-mt-id-en is smaller and simpler for Indonesian -> English.
TRANSLATION_MODEL_NAME = "Helsinki-NLP/opus-mt-id-en"
TRANSLATION_BATCH_SIZE_GPU = 16
TRANSLATION_BATCH_SIZE_CPU = 4
MAX_TRANSLATION_LENGTH = 256

# =========================
# Output files
# =========================
UNIQUE_CLEAN_FLAGS_PATH = OUTPUT_DIR / "01_shopee_unique_cleaned_noise_filter_flags.csv"
PASS_BEFORE_TRANSLATION_PATH = OUTPUT_DIR / "02_shopee_pass_noise_filtered_before_translation.csv"
FILTERED_OUT_AUDIT_PATH = OUTPUT_DIR / "03_shopee_filtered_out_noise_unrelated_audit.csv"
TRANSLATION_MAP_PATH = OUTPUT_DIR / "04_shopee_translation_map_noise_filtered.csv"
READY_FOR_DEBERTA_PATH = OUTPUT_DIR / "05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv"
READY_MINIMAL_PATH = OUTPUT_DIR / "05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter_minimal.csv"
EMPTY_AUTOLABEL_TEMPLATE_PATH = OUTPUT_DIR / "06_empty_autolabel_template_14aspect_no_aspect_filter.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "phase2_preprocess_filter_translate_14aspect_no_aspect_filter_summary.json"

print("RAW_CSV_PATH:", RAW_CSV_PATH)
print("RAW exists  :", RAW_CSV_PATH.exists())
print("OUTPUT_DIR :", OUTPUT_DIR)
print("Focused-aspect keyword filtering: OFF")
print("Minimum length filter:", MIN_REVIEW_WORDS, "words OR", MIN_REVIEW_CHARS, "characters")

RAW_CSV_PATH: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\shopee_google_play_reviews_raw.csv
RAW exists  : True
OUTPUT_DIR : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee
Focused-aspect keyword filtering: OFF
Minimum length filter: 3 words OR 12 characters


## 2. Define the exact Trustpilot 14-aspect schema

This is the schema used during the Trustpilot fine-tuning stage.

The original Trustpilot aspect columns use numeric labels:

```text
1.0 = aspect mentioned with Positive sentiment
0.0 = aspect mentioned with Negative sentiment
NaN = aspect not mentioned
```

For Shopee, we do **not** have ground-truth labels yet. This notebook only prepares the review text and schema for auto-labeling.

In [3]:
ASPECT_COLS = [
    "Information",
    "Product_features",
    "Item_quality",
    "App_experience",
    "Pricing",
    "Product_availability",
    "Payment",
    "Packaging",
    "Delivered_product_status",
    "Shipping",
    "Return_process",
    "Refund_process",
    "Customer_service",
    "Trust",
]

ASPECT_SENTIMENT_CLASSES = ["Negative", "Positive"]
GENERAL_TEXT_SENTIMENT_CLASSES = ["Negative", "Positive", "Mixed"]

TARGET_ASPECT_SCHEMA = "; ".join(ASPECT_COLS)
TARGET_ASPECT_SENTIMENT_SCHEMA = "; ".join(ASPECT_SENTIMENT_CLASSES)
TARGET_GENERAL_TEXT_SENTIMENT_SCHEMA = "; ".join(GENERAL_TEXT_SENTIMENT_CLASSES)

print("Number of aspects:", len(ASPECT_COLS))
print("Aspects:", ASPECT_COLS)
print("Aspect-level sentiment classes:", ASPECT_SENTIMENT_CLASSES)
print("General_text sentiment classes:", GENERAL_TEXT_SENTIMENT_CLASSES)

Number of aspects: 14
Aspects: ['Information', 'Product_features', 'Item_quality', 'App_experience', 'Pricing', 'Product_availability', 'Payment', 'Packaging', 'Delivered_product_status', 'Shipping', 'Return_process', 'Refund_process', 'Customer_service', 'Trust']
Aspect-level sentiment classes: ['Negative', 'Positive']
General_text sentiment classes: ['Negative', 'Positive', 'Mixed']


## 3. Load Shopee raw crawling dataset

In [4]:
if not RAW_CSV_PATH.exists():
    raise FileNotFoundError(f"Raw CSV not found: {RAW_CSV_PATH}")

raw_df = pd.read_csv(RAW_CSV_PATH)
if SMOKE_TEST_ROWS is not None:
    raw_df = raw_df.head(SMOKE_TEST_ROWS).copy()

print("Raw shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head())

# Auto-detect ID and review columns from common crawling output names.
ID_CANDIDATES = ["review_id_hash", "reviewId", "review_id", "id"]
REVIEW_CANDIDATES = ["content", "review", "review_text", "text", "translated_review"]

id_col = next((c for c in ID_CANDIDATES if c in raw_df.columns), None)
review_col = next((c for c in REVIEW_CANDIDATES if c in raw_df.columns), None)

if review_col is None:
    raise ValueError(f"Could not find review text column. Expected one of: {REVIEW_CANDIDATES}")

if id_col is None:
    print("No ID column found. A stable hash ID will be generated from review text.")
else:
    print("ID column:", id_col)

print("Review column:", review_col)

Raw shape: (15000, 14)
Columns: ['review_id_hash', 'content', 'score', 'thumbs_up_count', 'review_created_version', 'review_at', 'app_version', 'reply_content', 'replied_at', 'source_app_id', 'source_lang', 'source_country', 'crawl_timestamp_utc', 'weak_sentiment_from_star']


,review_id_hash,content,score,thumbs_up_count,review_created_version,review_at,app_version,reply_content,replied_at,source_app_id,source_lang,source_country,crawl_timestamp_utc,weak_sentiment_from_star
0,2d68668cf8c3d9b8,i hate your ads,1,0,3.75.28,2026-06-19T14:42:44,3.75.28,NaN,NaN,com.shopee.id,id,id,2026-06-20T07:44:28.937737+00:00,negative
1,d363df62267eae48,"shopee tolong ya biaya layanan jangan dinaikkan mulu, dari 1,000 lalu 2,000 sekarang 2.200. kalo bisa tolong balik lagi ke 1.000 ya. bikin ga mood belanja",1,0,3.76.24,2026-06-19T14:41:24,3.76.24,NaN,NaN,com.shopee.id,id,id,2026-06-20T07:44:28.937737+00:00,negative
2,4929519f6ef730a2,jelek,1,0,NaN,2026-06-19T14:40:51,NaN,NaN,NaN,com.shopee.id,id,id,2026-06-20T07:44:28.937737+00:00,negative
3,8bdfe8eba767c20d,aplikasi paling maksa selalu ada iklan pop dimana²,1,0,3.76.24,2026-06-19T14:39:49,3.76.24,NaN,NaN,com.shopee.id,id,id,2026-06-20T07:44:28.937737+00:00,negative
4,c6e103e312de60a4,"buka aplikasi, trus muncul penjualan live ini amat tidak menyenangkan, apalagi dengan volume full, udah diatur dari setelan, cuma tidak berhasil mohon bantu untuk menyelesaikan...",1,0,3.76.25,2026-06-19T14:39:20,3.76.25,"Hai kak Junaidi, mohon maaf atas ketidaknyamanannya. Terkait kendala yg kaka alami, agar bisa di cek lebih lanjut, mohon infokan detail kendalanya melalui DM ke Media Sosial ki...",2026-06-19T15:13:36,com.shopee.id,id,id,2026-06-20T07:44:28.937737+00:00,negative


ID column: review_id_hash
Review column: content


## 4. Cleaning functions


In [5]:
# Broad emoji/symbol pattern. This removes most pictographic emoji without deleting normal text.
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+",
    flags=re.UNICODE,
)

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}\b")
MENTION_PATTERN = re.compile(r"(?<!\w)@[A-Za-z0-9_\.]+")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEATED_CHAR_PATTERN = re.compile(r"(.)\1{4,}")

# Broken HTML/font-color fragments seen in crawled reviews.
# Examples handled: # FFFF00, #FFFF00, fontcolor="#FFFF00", font color="# FFFF00", &lt;font color="#FFFF00"&gt;.
BROKEN_HTML_OR_COLOR_PATTERNS = [
    re.compile(r"&(?:lt|gt|amp|quot|apos|nbsp);", re.IGNORECASE),
    re.compile(r"</?\s*(?:font|span|div|br|p|strong|em|b|i)[^>]*>", re.IGNORECASE),
    re.compile(r"(?:font\s*color|fontcolor|color|background-color)\s*=\s*['\"]?\s*#?\s*[0-9a-fA-F]{3,8}\s*['\"]?", re.IGNORECASE),
    re.compile(r"#\s*[0-9a-fA-F]{6}\s*['\"]?", re.IGNORECASE),
    re.compile(r"\b(?:fontcolor|font\s*color|background-color|fontsize)\b", re.IGNORECASE),
]

# Generic leftover HTML entities after html.unescape.
HTML_ENTITY_LEFTOVER_PATTERN = re.compile(r"&[a-zA-Z0-9#]+;")


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(str(text).encode("utf-8", errors="ignore")).hexdigest()[:length]


def has_broken_html_or_color_fragment(text) -> bool:
    if pd.isna(text):
        return False
    raw = str(text)
    fixed = ftfy_fix_text(raw)
    candidates = [raw, fixed, html.unescape(fixed)]
    return any(any(pat.search(candidate) for pat in BROKEN_HTML_OR_COLOR_PATTERNS) for candidate in candidates)


def clean_review_text(text) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = ftfy_fix_text(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    # Remove broken HTML/font-color fragments before regular HTML stripping.
    for pat in BROKEN_HTML_OR_COLOR_PATTERNS:
        text = pat.sub(" ", text)

    text = HTML_TAG_PATTERN.sub(" ", text)
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HTML_ENTITY_LEFTOVER_PATTERN.sub(" ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = text.replace("\u200b", " ").replace("\ufeff", " ")
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


def normalize_for_matching(text: str) -> str:
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = REPEATED_CHAR_PATTERN.sub(r"\1\1", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


def word_count(text: str) -> int:
    return len(str(text).split())


def has_probable_gibberish(text: str) -> bool:
    s = str(text).strip()
    if not s:
        return True
    compact = re.sub(r"\s+", "", s)

    # Long consonant clusters usually indicate gibberish/no-space broken text.
    if len(compact) >= 25 and re.search(r"[bcdfghjklmnpqrstvwxyz]{12,}", compact.lower()):
        return True

    # Too many symbols compared to letters/digits.
    if len(compact) >= 20:
        alpha = sum(ch.isalpha() for ch in compact)
        digit = sum(ch.isdigit() for ch in compact)
        if alpha + digit > 0:
            symbol_ratio = 1 - ((alpha + digit) / len(compact))
            if symbol_ratio > 0.45:
                return True

    # Very long no-space strings are often broken crawl/noise.
    if re.search(r"\b\S{45,}\b", s):
        return True

    # Repeated keyboard-like noise.
    if re.search(r"([a-z]{1,3})\1{5,}", compact.lower()):
        return True

    return False

## 5. Noise / unrelated-content filter rules


In [6]:
# Filtering happens before translation, so include Indonesian + English forms.
# These are only for removing obvious unrelated/noisy rows, NOT for focused-aspect selection.

SPAM_AD_TERMS = [
    "pinjaman online", "pinjol", "dana cepat", "saldo dana", "uang gratis", "pulsa gratis",
    "slot", "judi", "casino", "togel", "scatter", "maxwin", "jackpot", "bet", "taruhan",
    "whatsapp saya", "wa saya", "hubungi saya", "chat saya", "dm saya", "nomor wa", "wa admin",
    "promosi akun", "subscribe", "subrek", "follow ig", "follow instagram", "telegram", "join grup",
    "open bo", "bokep", "porn", "porno", "dewasa 18",
]

# Strong unrelated terms: filter only when they clearly indicate another app/topic or advertisement.
UNRELATED_STRONG_TERMS = [
    "free fire", "mobile legend", "mobile legends", "pubg", "roblox", "minecraft", "game ini",
    "youtube channel", "channel youtube", "tiktok saya", "instagram saya", "facebook saya",
    "lowongan kerja", "loker", "jasa tugas", "jasa skripsi", "jual akun", "beli akun",
    "politik", "pilpres", "capres", "caleg", "partai politik",
]

# Broad Shopee/e-commerce/app context for audit only. We do NOT require these to pass.
ECOMMERCE_OR_APP_CONTEXT_TERMS = [
    "shopee", "shopi", "shoppe", "aplikasi", "app", "apk", "belanja", "checkout", "keranjang",
    "pesanan", "order", "barang", "produk", "item", "seller", "penjual", "toko", "merchant",
    "pembayaran", "pay", "transfer", "cod", "spay", "shopeepay", "paylater", "voucher", "gratis ongkir",
    "ongkir", "pengiriman", "kirim", "dikirim", "kurir", "paket", "resi", "jnt", "jne", "sicepat", "anteraja",
    "refund", "retur", "return", "komplain", "cs", "customer service", "chat", "rating", "review",
]

SPAM_AD_TERMS_NORM = [normalize_for_matching(x) for x in SPAM_AD_TERMS]
UNRELATED_STRONG_TERMS_NORM = [normalize_for_matching(x) for x in UNRELATED_STRONG_TERMS]
ECOMMERCE_OR_APP_CONTEXT_TERMS_NORM = [normalize_for_matching(x) for x in ECOMMERCE_OR_APP_CONTEXT_TERMS]


def contains_phrase(norm_text: str, phrase: str) -> bool:
    if not phrase:
        return False
    # Word-boundary-ish matching for alphanumeric phrases.
    return re.search(r"(?<![a-z0-9])" + re.escape(phrase) + r"(?![a-z0-9])", norm_text) is not None


def has_any_phrase(norm_text: str, phrases: list[str]) -> bool:
    return any(contains_phrase(norm_text, kw) for kw in phrases)


def has_ecommerce_or_app_context(norm_text: str) -> bool:
    return has_any_phrase(norm_text, ECOMMERCE_OR_APP_CONTEXT_TERMS_NORM)


def is_probable_spam_or_unrelated_content(norm_text: str) -> bool:
    """Return True only for obvious spam/ads or clearly unrelated topics.

    Important: absence of e-commerce/aspect keywords is NOT enough to filter a review.
    """
    if not norm_text:
        return False

    if has_any_phrase(norm_text, SPAM_AD_TERMS_NORM):
        return True

    if has_any_phrase(norm_text, UNRELATED_STRONG_TERMS_NORM):
        return True

    # Contact/ad pattern: many numbers plus contact words, without Shopee/e-commerce context.
    has_contact_word = any(w in norm_text.split() for w in ["wa", "whatsapp", "telegram", "dm", "line"])
    digit_count = sum(ch.isdigit() for ch in norm_text)
    if has_contact_word and digit_count >= 6 and not has_ecommerce_or_app_context(norm_text):
        return True

    return False


# Safe column names for downstream auto-label template outputs.
def safe_col(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


ASPECT_PRED_COLS = {aspect: f"pred_aspect_{safe_col(aspect)}" for aspect in ASPECT_COLS}
ASPECT_PROB_COLS = {aspect: f"prob_aspect_{safe_col(aspect)}" for aspect in ASPECT_COLS}
ASPECT_SENTIMENT_COLS = {aspect: f"pred_sentiment_{safe_col(aspect)}" for aspect in ASPECT_COLS}

print("Noise/unrelated filter is ready.")
print("Focused-aspect filtering is OFF.")

Noise/unrelated filter is ready.
Focused-aspect filtering is OFF.


## 6. Apply cleaning, deduplication, and unrelated/noise filtering

In [7]:
df = raw_df.copy()

# Standard ID and text columns.
df["review_original"] = df[review_col].fillna("").astype(str)
df["had_broken_html_or_color_fragment"] = df["review_original"].progress_apply(has_broken_html_or_color_fragment)
df["review_clean"] = df["review_original"].progress_apply(clean_review_text)
df["review_match_norm"] = df["review_clean"].progress_apply(normalize_for_matching)
df["char_count_clean"] = df["review_clean"].str.len()
df["word_count_clean"] = df["review_clean"].apply(word_count)
df["review_text_hash"] = df["review_match_norm"].apply(stable_hash)

if id_col is not None:
    df["id"] = df[id_col].fillna("").astype(str)
    empty_id_mask = df["id"].str.len().eq(0)
    df.loc[empty_id_mask, "id"] = df.loc[empty_id_mask, "review_text_hash"]
else:
    df["id"] = df["review_text_hash"]

# Prefer newest duplicate when review_at exists, otherwise preserve original order.
df["_original_order"] = np.arange(len(df))
if "review_at" in df.columns:
    df["_review_datetime"] = pd.to_datetime(df["review_at"], errors="coerce", utc=True)
    df = df.sort_values(["_review_datetime", "_original_order"], ascending=[False, True]).copy()

# Unique reviews only by normalized cleaned review text.
df["is_duplicate_review_text"] = df.duplicated(subset=["review_text_hash"], keep="first")

# Filter flags. No aspect-keyword filtering is used here.
df["is_empty_after_cleaning"] = df["char_count_clean"].eq(0)
df["is_probable_gibberish"] = df["review_clean"].apply(has_probable_gibberish)
df["is_probable_spam_or_unrelated_content"] = df["review_match_norm"].apply(is_probable_spam_or_unrelated_content)
df["has_ecommerce_or_app_context"] = df["review_match_norm"].apply(has_ecommerce_or_app_context)

# Short reviews like "ok", "bagus", "mantap", "jelek" are usually not useful for ABSA auto-labeling.
df["is_too_short_review"] = (df["word_count_clean"] < MIN_REVIEW_WORDS) | (df["char_count_clean"] < MIN_REVIEW_CHARS)

# Final pass flag: unique + readable + not spam/unrelated + not too short.
df["pass_filter_for_deberta"] = (
    ~df["is_duplicate_review_text"]
    & ~df["is_empty_after_cleaning"]
    & ~df["is_probable_gibberish"]
    & ~df["is_probable_spam_or_unrelated_content"]
    & ~df["is_too_short_review"]
)


def build_filter_reason(row) -> str:
    reasons = []
    if row["is_duplicate_review_text"]:
        reasons.append("duplicate_review_text")
    if row["is_empty_after_cleaning"]:
        reasons.append("empty_after_cleaning")
    if row["is_probable_gibberish"]:
        reasons.append("probable_gibberish_or_broken_text")
    if row["is_probable_spam_or_unrelated_content"]:
        reasons.append("probable_spam_ad_or_unrelated_content")
    if row["is_too_short_review"]:
        reasons.append(f"too_short_review_under_{MIN_REVIEW_WORDS}_words_or_{MIN_REVIEW_CHARS}_chars")
    if row["had_broken_html_or_color_fragment"] and not reasons:
        # Not a drop reason by itself. This shows the row was cleaned but still passed.
        reasons.append("pass_cleaned_broken_html_or_color_fragment")
    return "; ".join(reasons) if reasons else "pass"


df["filter_reason"] = df.apply(build_filter_reason, axis=1)

# Attach schema metadata for traceability.
df["target_aspect_schema"] = TARGET_ASPECT_SCHEMA
df["target_aspect_sentiment_schema"] = TARGET_ASPECT_SENTIMENT_SCHEMA
df["target_general_text_sentiment_schema"] = TARGET_GENERAL_TEXT_SENTIMENT_SCHEMA
df["aspect_sentiment_rule"] = "Trustpilot aspect cells: 1.0=Positive, 0.0=Negative, NaN=not mentioned"
df["general_text_sentiment_rule"] = "General_text keeps original overall labels: Positive, Negative, Mixed"
df["filter_policy"] = "No focused-aspect keyword filtering; only duplicate, empty, short, gibberish, spam/ad, and clearly unrelated content filtering."

print("Raw rows:", len(raw_df))
print("Unique text rows:", int((~df["is_duplicate_review_text"]).sum()))
print("Pass rows:", int(df["pass_filter_for_deberta"].sum()))
print("Filtered out rows:", int((~df["pass_filter_for_deberta"]).sum()))
display(df[[
    "id", "review_original", "review_clean", "had_broken_html_or_color_fragment",
    "char_count_clean", "word_count_clean", "filter_reason", "pass_filter_for_deberta"
]].head(20))

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

Raw rows: 15000
Unique text rows: 11591
Pass rows: 9755
Filtered out rows: 5245


,id,review_original,review_clean,had_broken_html_or_color_fragment,char_count_clean,word_count_clean,filter_reason,pass_filter_for_deberta
0,2d68668cf8c3d9b8,i hate your ads,i hate your ads,False,15,4,pass,True
1,d363df62267eae48,"shopee tolong ya biaya layanan jangan dinaikkan mulu, dari 1,000 lalu 2,000 sekarang 2.200. kalo bisa tolong balik lagi ke 1.000 ya. bikin ga mood belanja","shopee tolong ya biaya layanan jangan dinaikkan mulu, dari 1,000 lalu 2,000 sekarang 2.200. kalo bisa tolong balik lagi ke 1.000 ya. bikin ga mood belanja",False,154,26,pass,True
2,4929519f6ef730a2,jelek,jelek,False,5,1,too_short_review_under_3_words_or_12_chars,False
3,8bdfe8eba767c20d,aplikasi paling maksa selalu ada iklan pop dimana²,aplikasi paling maksa selalu ada iklan pop dimana2,False,50,8,pass,True
4,c6e103e312de60a4,"buka aplikasi, trus muncul penjualan live ini amat tidak menyenangkan, apalagi dengan volume full, udah diatur dari setelan, cuma tidak berhasil mohon bantu untuk menyelesaikan...","buka aplikasi, trus muncul penjualan live ini amat tidak menyenangkan, apalagi dengan volume full, udah diatur dari setelan, cuma tidak berhasil mohon bantu untuk menyelesaikan...",False,190,27,pass,True
5,110b95ee33742273,"tolong dong iklannya di kurangin lagi baca watpaad tiba² pindah ke shopee, sudah di pencet tanda silang di atas malah ke shopee itu sangat menggangu pliss orang jg muak lama² l...","tolong dong iklannya di kurangin lagi baca watpaad tiba2 pindah ke shopee, sudah di pencet tanda silang di atas malah ke shopee itu sangat menggangu pliss orang jg muak lama2 l...",False,193,33,pass,True
6,c49004b4a142159c,"sangat puas dan asyik belanja dishoope, semoga shoope mangkin terdepan","sangat puas dan asyik belanja dishoope, semoga shoope mangkin terdepan",False,70,10,pass,True
7,52e7ed4683508031,"ada juga barang yg nyampai rusak, dan ada yg tidak sesuai dg pesanan","ada juga barang yg nyampai rusak, dan ada yg tidak sesuai dg pesanan",False,68,13,pass,True
8,45cc42de4e2ba534,"tolong iklannya jangan banyak"" atau tombol lewatinya di perbesar, saya lagi buat tugas di apk editor muncul iklan, lalu saya menekan lewati malah masuk di apk shopeenya alhasil...","tolong iklannya jangan banyak"" atau tombol lewatinya di perbesar, saya lagi buat tugas di apk editor muncul iklan, lalu saya menekan lewati malah masuk di apk shopeenya alhasil...",False,273,43,pass,True
9,83cdbca62c7d8b77,"terlalu banyak iklan nya sangat mengganggu, dikit² muncul iklan udh di pencet x nya tetap ke buka apl nya. tolong di perbaiki lagi","terlalu banyak iklan nya sangat mengganggu, dikit2 muncul iklan udh di pencet x nya tetap ke buka apl nya. tolong di perbaiki lagi",False,130,23,pass,True


## 7. Inspect filter summary

In [8]:
summary_counts = {
    "raw_rows": int(len(raw_df)),
    "unique_review_text_rows": int((~df["is_duplicate_review_text"]).sum()),
    "pass_filter_for_deberta_rows": int(df["pass_filter_for_deberta"].sum()),
    "filtered_out_rows": int((~df["pass_filter_for_deberta"]).sum()),
    "duplicate_review_text_rows": int(df["is_duplicate_review_text"].sum()),
    "empty_after_cleaning_rows": int(df["is_empty_after_cleaning"].sum()),
    "had_broken_html_or_color_fragment_rows": int(df["had_broken_html_or_color_fragment"].sum()),
    "probable_gibberish_rows": int(df["is_probable_gibberish"].sum()),
    "probable_spam_ad_or_unrelated_content_rows": int(df["is_probable_spam_or_unrelated_content"].sum()),
    "too_short_review_rows": int(df["is_too_short_review"].sum()),
}
summary_df = pd.DataFrame(summary_counts.items(), columns=["metric", "count"])
display(summary_df)

reason_counts_df = (
    df["filter_reason"]
    .value_counts(dropna=False)
    .rename_axis("filter_reason")
    .reset_index(name="count")
)
display(reason_counts_df)

# Quick audit previews.
filtered_preview_cols = ["id", "review_original", "review_clean", "char_count_clean", "word_count_clean", "filter_reason"]
print("Filtered-out preview:")
display(df.loc[~df["pass_filter_for_deberta"], filtered_preview_cols].head(30))

print("Rows that contained broken HTML/color fragments but were cleaned:")
display(df.loc[df["had_broken_html_or_color_fragment"], filtered_preview_cols + ["pass_filter_for_deberta"]].head(30))

,metric,count
0,raw_rows,15000
1,unique_review_text_rows,11591
2,pass_filter_for_deberta_rows,9755
3,filtered_out_rows,5245
4,duplicate_review_text_rows,3409
5,empty_after_cleaning_rows,112
6,had_broken_html_or_color_fragment_rows,0
7,probable_gibberish_rows,222
8,probable_spam_ad_or_unrelated_content_rows,139
9,too_short_review_rows,4709


,filter_reason,count
0,pass,9755
1,duplicate_review_text; too_short_review_under_3_words_or_12_chars,2961
2,too_short_review_under_3_words_or_12_chars,1601
3,duplicate_review_text,324
4,probable_spam_ad_or_unrelated_content,136
5,duplicate_review_text; empty_after_cleaning; probable_gibberish_or_broken_text; too_short_review_under_3_words_or_12_chars,111
6,probable_gibberish_or_broken_text,74
7,probable_gibberish_or_broken_text; too_short_review_under_3_words_or_12_chars,21
8,duplicate_review_text; probable_gibberish_or_broken_text; too_short_review_under_3_words_or_12_chars,13
9,probable_gibberish_or_broken_text; probable_spam_ad_or_unrelated_content,2


Filtered-out preview:


,id,review_original,review_clean,char_count_clean,word_count_clean,filter_reason
2,4929519f6ef730a2,jelek,jelek,5,1,too_short_review_under_3_words_or_12_chars
10,3e2f1bd57696938a,diskonyaaa kurang,diskonyaaa kurang,17,2,too_short_review_under_3_words_or_12_chars
11,411a6d0b31668463,baik,baik,4,1,too_short_review_under_3_words_or_12_chars
14,b85a1cd6d3337ae8,good job👍,good job,8,2,too_short_review_under_3_words_or_12_chars
22,79eaac9aa4335917,aplikasi tolol,aplikasi tolol,14,2,too_short_review_under_3_words_or_12_chars
30,8d592be09ca2acca,mantap 👍,mantap,6,1,too_short_review_under_3_words_or_12_chars
41,4739b35fdb4b6f53,trash,trash,5,1,too_short_review_under_3_words_or_12_chars
44,aa049edead7e7960,shpe tolol,shpe tolol,10,2,too_short_review_under_3_words_or_12_chars
48,d83dc901bd8915f7,wow,wow,3,1,too_short_review_under_3_words_or_12_chars
74,2e00839ed05ccc59,lumayan bagus,lumayan bagus,13,2,too_short_review_under_3_words_or_12_chars


Rows that contained broken HTML/color fragments but were cleaned:


,id,review_original,review_clean,char_count_clean,word_count_clean,filter_reason,pass_filter_for_deberta


## 8. Save pre-translation audit files

In [9]:
metadata_cols = [
    c for c in [
        "score", "thumbs_up_count", "review_created_version", "review_at", "app_version",
        "reply_content", "replied_at", "source_app_id", "source_lang", "source_country",
        "crawl_timestamp_utc", "weak_sentiment_from_star"
    ] if c in df.columns
]

base_cols = [
    "id", "review_original", "review_clean", "review_match_norm", "review_text_hash",
    "target_aspect_schema", "target_aspect_sentiment_schema", "target_general_text_sentiment_schema",
    "aspect_sentiment_rule", "general_text_sentiment_rule", "filter_policy",
    "has_ecommerce_or_app_context", "char_count_clean", "word_count_clean",
    "had_broken_html_or_color_fragment",
    "is_duplicate_review_text", "is_empty_after_cleaning", "is_probable_gibberish",
    "is_probable_spam_or_unrelated_content", "is_too_short_review",
    "pass_filter_for_deberta", "filter_reason",
]

export_cols = [c for c in base_cols + metadata_cols if c in df.columns]
unique_clean_flags_df = df[export_cols].copy()
pass_df = unique_clean_flags_df[unique_clean_flags_df["pass_filter_for_deberta"]].copy()
filtered_out_df = unique_clean_flags_df[~unique_clean_flags_df["pass_filter_for_deberta"]].copy()

unique_clean_flags_df.to_csv(UNIQUE_CLEAN_FLAGS_PATH, index=False, encoding="utf-8-sig")
pass_df.to_csv(PASS_BEFORE_TRANSLATION_PATH, index=False, encoding="utf-8-sig")
filtered_out_df.to_csv(FILTERED_OUT_AUDIT_PATH, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", UNIQUE_CLEAN_FLAGS_PATH)
print("-", PASS_BEFORE_TRANSLATION_PATH)
print("-", FILTERED_OUT_AUDIT_PATH)
print("Pass shape:", pass_df.shape)
print("Filtered shape:", filtered_out_df.shape)

Saved:
- C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\01_shopee_unique_cleaned_noise_filter_flags.csv
- C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\02_shopee_pass_noise_filtered_before_translation.csv
- C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\03_shopee_filtered_out_noise_unrelated_audit.csv
Pass shape: (9755, 34)
Filtered shape: (5245, 34)


## 9. Language detection


In [10]:
if LINGUA_AVAILABLE:
    detector = (
        LanguageDetectorBuilder
        .from_languages(
            Language.ENGLISH,
            Language.INDONESIAN,
            Language.MALAY
        )
        .with_preloaded_language_models()
        .build()
    )
else:
    detector = None

INDONESIAN_MARKERS = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "tidak", "gak", "ga", "nggak", "banget", "barang", "belanja", "aplikasi",
    "pengiriman", "pesanan", "seller", "penjual", "pembayaran", "harga", "ongkir", "paket"
}
ENGLISH_MARKERS = {
    "the", "and", "is", "are", "not", "very", "good", "bad", "item", "product", "shipping", "delivery", "payment", "price", "app"
}


def detect_language_label(text: str) -> str:
    text = str(text).strip()
    if not text:
        return "unknown"
    if detector is not None:
        try:
            lang = detector.detect_language_of(text)
            if lang is None:
                return "unknown"
            return lang.iso_code_639_1.name.lower()
        except Exception:
            pass
    # Fallback heuristic.
    tokens = set(normalize_for_matching(text).split())
    id_score = len(tokens & INDONESIAN_MARKERS)
    en_score = len(tokens & ENGLISH_MARKERS)
    if id_score > en_score:
        return "id"
    if en_score > id_score:
        return "en"
    # Most Shopee Indonesia reviews are Indonesian if uncertain.
    return "id"

pass_df["detected_language"] = pass_df["review_clean"].progress_apply(detect_language_label)
pass_df["should_translate"] = ~pass_df["detected_language"].isin(["en", "unknown"])

lang_counts = pass_df["detected_language"].value_counts(dropna=False).rename_axis("detected_language").reset_index(name="count")
display(lang_counts)
print("Rows to translate:", int(pass_df["should_translate"].sum()))

  0%|          | 0/9755 [00:00<?, ?it/s]

,detected_language,count
0,id,7732
1,ms,1864
2,en,159


Rows to translate: 9596


## 10. Translation to English


In [11]:
def clean_translated_text(text) -> str:
    text = clean_review_text(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if RUN_TRANSLATION:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = TRANSLATION_BATCH_SIZE_GPU if device == "cuda" else TRANSLATION_BATCH_SIZE_CPU
    print("Translation device:", device)
    print("Batch size:", batch_size)
    print("Model:", TRANSLATION_MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL_NAME)
    translation_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL_NAME).to(device)
    translation_model.eval()

    # Existing translation map cache.
    if TRANSLATION_MAP_PATH.exists():
        translation_map_df = pd.read_csv(TRANSLATION_MAP_PATH)
        translation_cache = dict(zip(translation_map_df["review_text_hash"].astype(str), translation_map_df["english_review"].astype(str)))
        print("Loaded existing translation cache:", len(translation_cache))
    else:
        translation_cache = {}

    def translate_batch(texts: list[str]) -> list[str]:
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_TRANSLATION_LENGTH,
        ).to(device)
        with torch.no_grad():
            generated = translation_model.generate(
                **inputs,
                max_length=MAX_TRANSLATION_LENGTH,
                num_beams=4,
                early_stopping=True,
            )
        outputs = tokenizer.batch_decode(generated, skip_special_tokens=True)
        return [clean_translated_text(x) for x in outputs]

    # Prepare unique texts needing translation and not cached.
    translate_source_df = (
        pass_df.loc[pass_df["should_translate"], ["review_text_hash", "review_clean"]]
        .drop_duplicates(subset=["review_text_hash"])
        .copy()
    )
    translate_source_df = translate_source_df[~translate_source_df["review_text_hash"].astype(str).isin(translation_cache.keys())].copy()
    print("New unique texts to translate:", len(translate_source_df))

    hashes = translate_source_df["review_text_hash"].astype(str).tolist()
    texts = translate_source_df["review_clean"].astype(str).tolist()

    for start in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch_hashes = hashes[start:start + batch_size]
        batch_texts = texts[start:start + batch_size]
        try:
            batch_outputs = translate_batch(batch_texts)
        except Exception as e:
            print(f"Translation batch failed at {start}: {e}")
            # Fallback: keep original text so the row is not lost.
            batch_outputs = batch_texts
        for h, en in zip(batch_hashes, batch_outputs):
            translation_cache[h] = en

    # Apply English text: English passthrough for already-English reviews; translated text for others.
    def get_english_review(row) -> str:
        if not row["should_translate"]:
            return clean_translated_text(row["review_clean"])
        return clean_translated_text(translation_cache.get(str(row["review_text_hash"]), row["review_clean"]))

    pass_df["english_review"] = pass_df.apply(get_english_review, axis=1)

    # Save translation map.
    translation_map_records = []
    for h, en in translation_cache.items():
        translation_map_records.append({"review_text_hash": h, "english_review": en})
    translation_map_df = pd.DataFrame(translation_map_records).drop_duplicates(subset=["review_text_hash"])
    translation_map_df.to_csv(TRANSLATION_MAP_PATH, index=False, encoding="utf-8-sig")
    print("Saved translation map:", TRANSLATION_MAP_PATH)

else:
    print("RUN_TRANSLATION=False, using cleaned original text as english_review placeholder.")
    pass_df["english_review"] = pass_df["review_clean"].apply(clean_translated_text)
    pass_df["translation_status"] = "not_translated_run_translation_false"

if "translation_status" not in pass_df.columns:
    pass_df["translation_status"] = np.where(pass_df["should_translate"], "translated", "english_passthrough")

pass_df["english_review"] = pass_df["english_review"].fillna("").astype(str).apply(clean_translated_text)
pass_df["english_review_word_count"] = pass_df["english_review"].apply(word_count)
pass_df["english_review_char_count"] = pass_df["english_review"].str.len()

print("Ready after translation rows:", int(pass_df["english_review_char_count"].gt(0).sum()))
display(pass_df[["id", "review_clean", "detected_language", "should_translate", "english_review"]].head(20))

Translation device: cuda
Batch size: 16
Model: Helsinki-NLP/opus-mt-id-en


c:\venvs\skripsi-gpu311\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

New unique texts to translate: 9596


Translating:   0%|          | 0/600 [00:00<?, ?it/s]

Saved translation map: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\04_shopee_translation_map_noise_filtered.csv
Ready after translation rows: 9755


,id,review_clean,detected_language,should_translate,english_review
0,2d68668cf8c3d9b8,i hate your ads,en,False,i hate your ads
1,d363df62267eae48,"shopee tolong ya biaya layanan jangan dinaikkan mulu, dari 1,000 lalu 2,000 sekarang 2.200. kalo bisa tolong balik lagi ke 1.000 ya. bikin ga mood belanja",id,True,"Shopee please do not raise the service fee, from 1,000 then 2,000 now 2,200. If you could please go back to 1,000."
3,8bdfe8eba767c20d,aplikasi paling maksa selalu ada iklan pop dimana2,id,True,The most maximal apps have pop ads everywhere.
4,c6e103e312de60a4,"buka aplikasi, trus muncul penjualan live ini amat tidak menyenangkan, apalagi dengan volume full, udah diatur dari setelan, cuma tidak berhasil mohon bantu untuk menyelesaikan...",id,True,"Open the application, and then there's this live sale. It's not fun, especially with the full volume, set from the suit, just couldn't help. Please help me solve my problem."
5,110b95ee33742273,"tolong dong iklannya di kurangin lagi baca watpaad tiba2 pindah ke shopee, sudah di pencet tanda silang di atas malah ke shopee itu sangat menggangu pliss orang jg muak lama2 l...",id,True,"please faddnotagain read watpaad suddento moveto shopee,havestit crossedup buttothe shopee it'sterrible surresistible respectiatepeopletoo long seethe shopeeadvert"
6,c49004b4a142159c,"sangat puas dan asyik belanja dishoope, semoga shoope mangkin terdepan",id,True,"So satisfied and busy shopping in a diphoope, may the first-rate shooter"
7,52e7ed4683508031,"ada juga barang yg nyampai rusak, dan ada yg tidak sesuai dg pesanan",id,True,"There's also broken stuff, and something doesn't match the order."
8,45cc42de4e2ba534,"tolong iklannya jangan banyak"" atau tombol lewatinya di perbesar, saya lagi buat tugas di apk editor muncul iklan, lalu saya menekan lewati malah masuk di apk shopeenya alhasil...",id,True,"Please don't do too much commercial"" or button passes zoomed in, I'm working on an apk editor show up an ad, and then I press through, and then I go through the apk shopee, and..."
9,83cdbca62c7d8b77,"terlalu banyak iklan nya sangat mengganggu, dikit2 muncul iklan udh di pencet x nya tetap ke buka apl nya. tolong di perbaiki lagi",id,True,"Too many ads. It's annoying, a little bit of an udh ad on the dial x. Keep it open."
12,bb2df138ba17108d,paansi gua ada di aplikasi manapun kalo iklan shoope trs yg muncul,id,True,My paansi is on any application if the shoope trs ad appears


In [13]:
def final_clean_english_review(text):
    s = str(text)

    # Remove HTML tags and escaped HTML entities
    s = re.sub(r"&lt;[^&]*?&gt;|<[^>]+>", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"&(?:amp|lt|gt|quot|apos);|&#\d+;", " ", s, flags=re.IGNORECASE)

    # Remove broken font/color fragments such as FFFF00, #FFFF00, fontcolor00, font"
    s = re.sub(r'font\s*color\s*=\s*["\']?\s*#?\s*f{4}00\s*["\']?', " ", s, flags=re.IGNORECASE)
    s = re.sub(r'fontcolor\s*=?\s*["\']?\s*#?\s*f{4}00?\s*["\']?', " ", s, flags=re.IGNORECASE)
    s = re.sub(r"#?\s*f{4}00\s*['\"]?", " ", s, flags=re.IGNORECASE)
    s = re.sub(r'\bfontcolor\d*\b|\bfont\b\s*["\']?', " ", s, flags=re.IGNORECASE)

    # Remove leftover translation/html artifacts like =" =" =" 
    s = re.sub(r'(?:=\s*["\']?\s*){2,}', " ", s)
    s = re.sub(r'\s*=\s*["\']?\s*', " ", s)

    # Normalize spacing
    s = re.sub(r"\s+", " ", s).strip()
    return s


def count_words(text):
    return len(re.findall(r"\b[\w']+\b", str(text)))


def is_broken_or_gibberish(text):
    s = str(text).lower()
    tokens = re.findall(r"[a-zA-Z]+", s)

    if not tokens:
        return True

    # Long glued words usually come from broken translation:
    # example: "alreadytransfertosopi", "ifthereavoucheralwayscannot"
    has_long_glued_token = any(len(t) >= 20 for t in tokens)

    # Repeated nonsense patterns:
    # example: "kkkkkkkkkk", "pekekekekekekekekekekapae"
    has_repeated_syllable = re.search(r"([a-z]{2,5})\1{3,}", s) is not None

    # Many consonant-heavy weird tokens
    no_vowel_tokens = [
        t for t in tokens
        if len(t) >= 6 and not re.search(r"[aeiou]", t)
    ]

    weird_ratio = (
        int(has_long_glued_token)
        + int(has_repeated_syllable)
        + len(no_vowel_tokens)
    ) / max(len(tokens), 1)

    return has_long_glued_token or has_repeated_syllable or weird_ratio >= 0.25


def has_broken_html_or_color(text):
    s = str(text).lower()
    return bool(re.search(
        r"(font\s*color|fontcolor|#?\s*f{4}00|ffff00|=\"\s*=\"|font\")",
        s
    ))


def is_spam_or_unrelated(text):
    s = str(text).lower()

    # Keep this strict so valid Shopee complaints are not removed accidentally.
    spam_patterns = [
        r"https?://",
        r"www\.",
        r"\btelegram\b",
        r"\bslot\b",
        r"\bcasino\b",
        r"\bporn\b",
        r"\bjud[iy]\b",
        r"\bpinjol\b",
        r"\bearn money\b",
        r"\bfree money\b",
        r"\bpromo code\b",
        r"\bfollow my\b",
    ]

    return any(re.search(pattern, s) for pattern in spam_patterns)

## 11. Create DeBERTa-ready datasets and empty auto-label template

In [14]:
# ============================================================
# Final post-translation quality filter
# Output: id + english_review only
# ============================================================

ready_df = pass_df.copy()

# Clean after translation
ready_df["english_review"] = ready_df["english_review"].astype(str).map(final_clean_english_review)

# Basic quality features
ready_df["final_word_count"] = ready_df["english_review"].map(count_words)
ready_df["final_char_len"] = ready_df["english_review"].str.len()

# Final filter reasons
ready_df["final_filter_reason"] = "keep"

ready_df.loc[
    ready_df["english_review"].str.strip().eq(""),
    "final_filter_reason"
] = "empty_after_final_cleaning"

ready_df.loc[
    (ready_df["final_filter_reason"] == "keep") &
    (
        (ready_df["final_word_count"] < 4) |
        (ready_df["final_char_len"] < 15)
    ),
    "final_filter_reason"
] = "too_short"

ready_df.loc[
    (ready_df["final_filter_reason"] == "keep") &
    ready_df["english_review"].map(has_broken_html_or_color),
    "final_filter_reason"
] = "broken_html_or_color_fragment"

ready_df.loc[
    (ready_df["final_filter_reason"] == "keep") &
    ready_df["english_review"].map(is_broken_or_gibberish),
    "final_filter_reason"
] = "gibberish_or_broken_translation"

ready_df.loc[
    (ready_df["final_filter_reason"] == "keep") &
    ready_df["english_review"].map(is_spam_or_unrelated),
    "final_filter_reason"
] = "spam_or_unrelated"

# Remove duplicate English reviews after final cleaning
duplicate_english_mask = ready_df["english_review"].str.lower().str.strip().duplicated(keep="first")

ready_df.loc[
    (ready_df["final_filter_reason"] == "keep") &
    duplicate_english_mask,
    "final_filter_reason"
] = "duplicate_english_review"

# Final pass-only DeBERTa input
ready_export_df = ready_df.loc[
    ready_df["final_filter_reason"] == "keep",
    ["id", "english_review"]
].copy()

# Audit file for removed reviews
final_filtered_out_df = ready_df.loc[
    ready_df["final_filter_reason"] != "keep",
    ["id", "english_review", "final_filter_reason"]
].copy()

ready_export_df.to_csv(READY_FOR_DEBERTA_PATH, index=False, encoding="utf-8-sig")
ready_export_df.to_csv(READY_MINIMAL_PATH, index=False, encoding="utf-8-sig")

FINAL_FILTERED_OUT_PATH = OUTPUT_DIR / "06_final_filtered_out_after_translation_audit.csv"
final_filtered_out_df.to_csv(FINAL_FILTERED_OUT_PATH, index=False, encoding="utf-8-sig")

print("Final DeBERTa-ready rows:", len(ready_export_df))
print("Final filtered-out rows :", len(final_filtered_out_df))
print("Final columns:", ready_export_df.columns.tolist())
print("Saved:", READY_FOR_DEBERTA_PATH)
print("Saved audit:", FINAL_FILTERED_OUT_PATH)

display(ready_export_df.head())
display(final_filtered_out_df["final_filter_reason"].value_counts())

Final DeBERTa-ready rows: 8179
Final filtered-out rows : 1576
Final columns: ['id', 'english_review']
Saved: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv
Saved audit: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\06_final_filtered_out_after_translation_audit.csv


,id,english_review
0,2d68668cf8c3d9b8,i hate your ads
1,d363df62267eae48,"Shopee please do not raise the service fee, from 1,000 then 2,000 now 2,200. If you could please go back to 1,000."
3,8bdfe8eba767c20d,The most maximal apps have pop ads everywhere.
4,c6e103e312de60a4,"Open the application, and then there's this live sale. It's not fun, especially with the full volume, set from the suit, just couldn't help. Please help me solve my problem."
6,c49004b4a142159c,"So satisfied and busy shopping in a diphoope, may the first-rate shooter"


final_filter_reason
too_short                          1180
gibberish_or_broken_translation     279
duplicate_english_review            116
spam_or_unrelated                     1
Name: count, dtype: int64

## 12. Save run summary

In [ ]:
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_csv_path": str(RAW_CSV_PATH),
    "output_dir": str(OUTPUT_DIR),
    "raw_rows": int(len(raw_df)),
    "unique_review_text_rows": int((~df["is_duplicate_review_text"]).sum()),
    "pass_filter_for_deberta_rows": int(df["pass_filter_for_deberta"].sum()),
    "ready_for_deberta_rows": int(len(ready_df)),
    "filtered_out_rows": int((~df["pass_filter_for_deberta"]).sum()),
    "focused_aspect_keyword_filtering": False,
    "min_review_words": int(MIN_REVIEW_WORDS),
    "min_review_chars": int(MIN_REVIEW_CHARS),
    "filter_policy": "No focused-aspect keyword filtering; only duplicate, empty, short, gibberish, spam/ad, and clearly unrelated content filtering.",
    "aspect_schema_count": len(ASPECT_COLS),
    "aspect_schema": ASPECT_COLS,
    "aspect_level_sentiment_classes": ASPECT_SENTIMENT_CLASSES,
    "general_text_sentiment_classes": GENERAL_TEXT_SENTIMENT_CLASSES,
    "translation_model_name": TRANSLATION_MODEL_NAME if RUN_TRANSLATION else None,
    "run_translation": bool(RUN_TRANSLATION),
    "output_files": {
        "unique_clean_flags": str(UNIQUE_CLEAN_FLAGS_PATH),
        "pass_before_translation": str(PASS_BEFORE_TRANSLATION_PATH),
        "filtered_out_audit": str(FILTERED_OUT_AUDIT_PATH),
        "translation_map": str(TRANSLATION_MAP_PATH),
        "ready_for_deberta_full": str(READY_FOR_DEBERTA_PATH),
        "ready_for_deberta_minimal": str(READY_MINIMAL_PATH),
        "empty_autolabel_template": str(EMPTY_AUTOLABEL_TEMPLATE_PATH),
    },
}

with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Saved summary:", SUMMARY_JSON_PATH)

## 13. Which file goes to DeBERTa auto-labeling?

Recommended file:

```text
05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter_minimal.csv
```

Main columns:

```text
id
english_review
```

The downstream auto-labeling notebook should predict:

```text
14 aspect detection labels:
Information, Product_features, Item_quality, App_experience, Pricing, Product_availability, Payment, Packaging, Delivered_product_status, Shipping, Return_process, Refund_process, Customer_service, Trust

Aspect-level sentiment:
Positive, Negative

General_text sentiment:
Positive, Negative, Mixed
```